# Deep Hedging in Black-Scholes: Coarse vs Fine Steps (TensorFlow Backend)
This notebook trains neural network deep hedging agents using TensorFlow/Keras on two time discretizations:
1. `trainer_bs_coarse`: 20 weekly time steps ($dt = 0.02$, batch size = 2000)
2. `trainer_bs_fine`: 100 daily time steps ($dt = 0.004$, batch size = 2000)

**Backend**: TensorFlow 2 / Keras.
**Self-contained**: Uses local `dh_tf` module.


In [1]:
import os
os.environ["DH_MODE"] = "smoke"
os.environ.pop("DH_EPOCHS", None)
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import sys
import tensorflow as tf

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("Mode:", os.environ["DH_MODE"])


Python: D:\deephedging-rewritten\.venv-tf\Scripts\python.exe
TensorFlow: 2.15.1
Mode: smoke


In [2]:
import os, sys
from pathlib import Path

NB_DIR = Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from cdxbasics.config import Config
from IPython.display import display, Markdown

from dh_tf.world import SimpleWorld_Spot_ATM
from dh_tf.gym import VanillaDeepHedgingGym
from dh_tf.trainer import train
from configs.catalog import get_config

MODE = os.environ.get("DH_MODE", "original")
def get_epochs(default_ep):
    if os.environ.get("DH_EPOCHS"):
        return int(os.environ.get("DH_EPOCHS"))
    return 2 if MODE == "smoke" else default_ep

def get_samples(default_samples):
    return min(default_samples, 500) if MODE == "smoke" else default_samples

print(f"trainer-bs.ipynb (TF) initialized | Mode: {MODE}")


Tensorflow version 2.15.1 running on 1 CPUs and 0 GPUs



trainer-bs.ipynb (TF) initialized | Mode: smoke


## 1. `trainer_bs_coarse`: 20 Time Steps ($dt = 0.02$)


In [3]:
cfg_coarse = get_config("trainer_bs_coarse", mode=MODE)
n_samples = get_samples(cfg_coarse["samples"])
n_epochs = get_epochs(cfg_coarse["epochs"])
batch_size = min(cfg_coarse["batch_size"], n_samples)

config = Config()
config.world.samples = n_samples
config.world.steps = cfg_coarse["steps"]
config.world.dt = cfg_coarse["dt"]
config.world.drift = cfg_coarse["drift"]
config.world.cost_s = cfg_coarse["cost_s"]
config.world.black_scholes = True

config.gym.objective.utility = cfg_coarse["utility"]
config.gym.objective.lmbda = cfg_coarse["lmbda"]
config.gym.agent.network.depth = cfg_coarse["depth"]
config.gym.agent.network.width = cfg_coarse["width"]
config.gym.agent.network.activation = cfg_coarse["activation"]

config.trainer.train.optimizer.name = "adam"
config.trainer.train.optimizer.learning_rate = cfg_coarse["lr"]
config.trainer.train.batch_size = batch_size
config.trainer.train.epochs = n_epochs
config.trainer.caching.mode = "off"
config.trainer.output_level = "all" if MODE == "original" else "quiet"

print(f"Training trainer_bs_coarse: {n_samples} samples, {cfg_coarse['steps']} steps, {n_epochs} epochs...")
world_coarse = SimpleWorld_Spot_ATM(config.world)
val_world_coarse = world_coarse.clone(samples=max(n_samples // 2, 50))
gym_coarse = VanillaDeepHedgingGym(config.gym)

train(gym=gym_coarse, world=world_coarse, val_world=val_world_coarse, config=config.trainer)
print("Finished trainer_bs_coarse training.")


Training trainer_bs_coarse: 500 samples, 20 steps, 2 epochs...



Finished trainer_bs_coarse training.


## 2. `trainer_bs_fine`: 100 Time Steps ($dt = 0.004$)


In [4]:
cfg_fine = get_config("trainer_bs_fine", mode=MODE)
n_samples = get_samples(cfg_fine["samples"])
n_epochs = get_epochs(cfg_fine["epochs"])
batch_size = min(cfg_fine["batch_size"], n_samples)

config_fine = Config()
config_fine.world.samples = n_samples
config_fine.world.steps = cfg_fine["steps"]
config_fine.world.dt = cfg_fine["dt"]
config_fine.world.drift = cfg_fine["drift"]
config_fine.world.cost_s = cfg_fine["cost_s"]
config_fine.world.black_scholes = True

config_fine.gym.objective.utility = cfg_fine["utility"]
config_fine.gym.objective.lmbda = cfg_fine["lmbda"]
config_fine.gym.agent.network.depth = cfg_fine["depth"]
config_fine.gym.agent.network.width = cfg_fine["width"]
config_fine.gym.agent.network.activation = cfg_fine["activation"]

config_fine.trainer.train.optimizer.name = "adam"
config_fine.trainer.train.optimizer.learning_rate = cfg_fine["lr"]
config_fine.trainer.train.batch_size = batch_size
config_fine.trainer.train.epochs = n_epochs
config_fine.trainer.caching.mode = "off"
config_fine.trainer.output_level = "all" if MODE == "original" else "quiet"

print(f"Training trainer_bs_fine: {n_samples} samples, {cfg_fine['steps']} steps, {n_epochs} epochs...")
world_fine = SimpleWorld_Spot_ATM(config_fine.world)
val_world_fine = world_fine.clone(samples=max(n_samples // 2, 50))
gym_fine = VanillaDeepHedgingGym(config_fine.gym)

train(gym=gym_fine, world=world_fine, val_world=val_world_fine, config=config_fine.trainer)
print("Finished trainer_bs_fine training.")


Training trainer_bs_fine: 500 samples, 100 steps, 2 epochs...
Finished trainer_bs_fine training.


### Summary
Both coarse and fine Black-Scholes models completed training successfully.
